In [1]:
# check for valid gpu
!nvidia-smi

Wed Aug 13 06:48:03 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.94                 Driver Version: 560.94         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3070      WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   47C    P8             19W /  220W |    2185MiB /   8192MiB |      1%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# establish home directory for project
import os
home = os.getcwd()
home

'c:\\Users\\reece\\Programming\\ms\\bird'

In [3]:
# check ultralytics version
import ultralytics
from ultralytics import YOLO
ultralytics.checks()

Ultralytics 8.3.162  Python-3.9.23 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
Setup complete  (12 CPUs, 31.1 GB RAM, 437.1/464.9 GB disk)


### CLI video inference

In [4]:
# run inference on content of video directory
%cd {home}
!yolo task=detect mode=track model={home}/runs/detect/train3/weights/best.pt source={home}/videos

c:\Users\reece\Programming\ms\bird
Ultralytics 8.3.162  Python-3.9.23 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
YOLOv12n summary (fused): 159 layers, 2,558,093 parameters, 0 gradients, 6.3 GFLOPs

video 1/11 (frame 1/1801) c:\Users\reece\Programming\ms\bird\videos\2025-02-15_14-53-45_HENSH_A.mp4: 384x640 1 feeder, 1 j, 48.2ms
video 1/11 (frame 2/1801) c:\Users\reece\Programming\ms\bird\videos\2025-02-15_14-53-45_HENSH_A.mp4: 384x640 1 feeder, 1 j, 10.0ms
video 1/11 (frame 3/1801) c:\Users\reece\Programming\ms\bird\videos\2025-02-15_14-53-45_HENSH_A.mp4: 384x640 1 feeder, 1 j, 8.8ms
video 1/11 (frame 4/1801) c:\Users\reece\Programming\ms\bird\videos\2025-02-15_14-53-45_HENSH_A.mp4: 384x640 1 feeder, 1 j, 8.7ms
video 1/11 (frame 5/1801) c:\Users\reece\Programming\ms\bird\videos\2025-02-15_14-53-45_HENSH_A.mp4: 384x640 1 feeder, 1 j, 8.8ms
video 1/11 (frame 6/1801) c:\Users\reece\Programming\ms\bird\videos\2025-02-15_14-53-45_HENSH_A.mp4: 384x640 1 feeder, 1 j, 9.0ms
vid

### Python per-frame inference

In [16]:
# load pre-trained model
model = YOLO(f"{home}/runs/detect/train3/weights/best.pt")
# set source for inference
video = ("2025-03-03_08-07-19_HENSH_D")
source = (f"{home}/videos/{video}.mp4")
# run inference
results = model.track(source=source, stream=True)

In [17]:
import pandas as pd
# empty dataframe to store results
df = pd.DataFrame()
# columns=['x1', 'y1', 'x2', 'y2', 'id', 'confidence', 'class']
for r in results:
    if r.boxes is not None and len(r.boxes) > 0:
        # convert the boxes to a dataframe and concat with the main dataframe
        boxes_df = pd.DataFrame(r.boxes.data.tolist())
        df = pd.concat([df, boxes_df], ignore_index=True)


video 1/1 (frame 1/2701) c:\Users\reece\Programming\ms\bird\videos\2025-03-03_08-07-19_HENSH_D.mp4: 384x640 1 feeder, 1 gt, 39.6ms
video 1/1 (frame 2/2701) c:\Users\reece\Programming\ms\bird\videos\2025-03-03_08-07-19_HENSH_D.mp4: 384x640 1 feeder, 1 gt, 38.9ms
video 1/1 (frame 3/2701) c:\Users\reece\Programming\ms\bird\videos\2025-03-03_08-07-19_HENSH_D.mp4: 384x640 1 feeder, 1 gt, 16.8ms
video 1/1 (frame 4/2701) c:\Users\reece\Programming\ms\bird\videos\2025-03-03_08-07-19_HENSH_D.mp4: 384x640 1 feeder, 1 gt, 14.9ms
video 1/1 (frame 5/2701) c:\Users\reece\Programming\ms\bird\videos\2025-03-03_08-07-19_HENSH_D.mp4: 384x640 1 feeder, 1 gt, 16.6ms
video 1/1 (frame 6/2701) c:\Users\reece\Programming\ms\bird\videos\2025-03-03_08-07-19_HENSH_D.mp4: 384x640 1 feeder, 1 gt, 16.3ms
video 1/1 (frame 7/2701) c:\Users\reece\Programming\ms\bird\videos\2025-03-03_08-07-19_HENSH_D.mp4: 384x640 1 feeder, 1 gt, 15.4ms
video 1/1 (frame 8/2701) c:\Users\reece\Programming\ms\bird\videos\2025-03-03_08-0

In [18]:
# class name values
model.names

{0: 'bt', 1: 'feeder', 2: 'gt', 3: 'j', 4: 'lt', 5: 'r', 6: 'squirrel'}

In [ ]:
# rename columns
df.rename(columns={0: 'x1', 1: 'y1', 2: 'x2', 3: 'y2', 4: 'id', 5: 'confidence', 6: 'class'}, inplace=True)
# preview dataframe
df

,x1,y1,x2,y2,id,confidence,class
0,801.237793,904.519775,2051.123535,1272.723389,1.0,0.953495,1.0
1,1249.280762,814.459839,1621.296875,1136.113770,2.0,0.920419,2.0
2,801.322327,905.234070,2051.131348,1272.818359,1.0,0.953183,1.0
3,1250.261230,814.888184,1615.936157,1135.936401,2.0,0.916634,2.0
4,801.456238,906.602356,2051.034912,1273.406860,1.0,0.952286,1.0
...,...,...,...,...,...,...,...
4298,794.793457,895.345154,2053.834717,1261.085571,1.0,0.956797,1.0
4299,794.683716,895.278992,2053.767090,1261.085571,1.0,0.956697,1.0
4300,794.664185,895.343445,2053.803467,1261.189331,1.0,0.956743,1.0
4301,794.710266,895.328552,2053.855713,1261.204468,1.0,0.956718,1.0


In [22]:
# save results to csv
df.to_csv(f"{home}/results/{video}_results.csv", index=False, header=True)